# Seto-Small (~500M): Pretraining на Kaggle 2x T4

**Цель:** Получить базовую двуязычную модель `Seto-Small` (~490M параметров: 24 слоя, d_model=1280, 20Q/5KV голов, GQA + SwiGLU + RoPE).

### Баланс языков:
- **Русский (~45%)**: FineWeb2 `rus_Cyrl` + Wikipedia RU
- **Английский (~45%)**: FineWeb `fineweb_100BT`
- **Украинский (~7%)**: FineWeb2 `ukr_Cyrl`
- **Код / логика (~3%)**: Python (GitHub code)

### Режимы (`RUN_MODE`):
- `smoke`: быстрый тест (1 000 шагов, ~32M токенов) для проверки отсутствия ошибок и утечек памяти.
- `full`: полноценный pretrain (~1.2–1.5 млрд токенов). Запускается в 3-4 сессии Kaggle по 10-12 часов с автопродолжением.

In [ ]:
!pip install -q tokenizers datasets

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path
import torch

WORK = Path("/kaggle/working") if Path("/kaggle").exists() else Path("/content")
RUN_MODE = "full"  # "smoke" для быстрой проверки пайплайна или "full" для основного обучения
GPU_COUNT = max(1, torch.cuda.device_count())

if RUN_MODE == "smoke":
    MAX_SAMPLES_RU = 20_000
    MAX_SAMPLES_EN = 20_000
    MAX_SAMPLES_UK = 3_000
    MAX_SAMPLES_WIKI = 2_000
    MAX_SAMPLES_CODE = 2_000
    MAX_STEPS = 1_000
    SAVE_EVERY = 250
else:
    # Сбалансированный корпус (~1.2B токенов)
    MAX_SAMPLES_RU = 400_000
    MAX_SAMPLES_EN = 400_000
    MAX_SAMPLES_UK = 60_000
    MAX_SAMPLES_WIKI = 30_000
    MAX_SAMPLES_CODE = 20_000
    MAX_STEPS = 40_000
    SAVE_EVERY = 500

REPO = WORK / "seto"
HF_CACHE = WORK / "seto-hf-cache"
TOKENIZER = WORK / "seto-tokenizer"
DATA = WORK / f"seto-small-data-{RUN_MODE}"
OUTPUT = WORK / f"seto-small-1-{RUN_MODE}"
DATA_READY = DATA / ".ready"

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_DISABLE_XET"] = "1"

def run(*args, cwd=None):
    print("+", " ".join(map(str, args)), flush=True)
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

print(f"Режим: {RUN_MODE}")
print(f"GPU: {GPU_COUNT} x {torch.cuda.get_device_name(0) if torch.cuda.is_available() else CPU}")
print(f"Папка вывода: {OUTPUT}")

In [ ]:
# Клонирование / синхронизация репозитория Seto
if REPO.exists():
    run("git", "pull", "--ff-only", cwd=REPO)
else:
    run("git", "clone", "https://github.com/mosshaven/seto.git", REPO)

sys.path.insert(0, str(REPO))
print("Репозиторий готов:", REPO)

In [ ]:
import numpy as np

def shard_token_count():
    return sum(
        path.stat().st_size // np.dtype(np.uint16).itemsize
        for path in sorted((DATA / "shards").glob("*.bin"))
    )

minimum_tokens = max(1_000_000, (MAX_SAMPLES_RU + MAX_SAMPLES_EN) * 200)
existing_tokens = shard_token_count()

if existing_tokens >= minimum_tokens and (TOKENIZER / "tokenizer.json").exists():
    DATA_READY.touch()
    print(f"Используем существующие шарды: {existing_tokens:,} токенов в {DATA / shards}.")
elif not DATA_READY.exists():
    command = [
        sys.executable, "-u", "scripts/prepare_data.py",
        "--output-dir", DATA,
        "--tokenizer-dir", TOKENIZER,
        "--max-samples-ru", MAX_SAMPLES_RU,
        "--max-samples-en", MAX_SAMPLES_EN,
        "--max-samples-uk", MAX_SAMPLES_UK,
        "--max-samples-wiki", MAX_SAMPLES_WIKI,
        "--max-samples-technical", MAX_SAMPLES_CODE,
        "--shard-size", "100000000",
    ]
    if (TOKENIZER / "tokenizer.json").exists():
        command.append("--skip-tokenizer")

    try:
        run(*command, cwd=REPO)
    except subprocess.CalledProcessError as error:
        recovered = shard_token_count()
        if recovered < 10_000_000:
            raise RuntimeError(f"Слишком мало токенов скачано: {recovered:,}") from error
        print(f"Поток прерван, но сохранено {recovered:,} токенов. Продолжаем на них.")

    DATA_READY.touch()
else:
    print("Данные уже готовы:", DATA)

shards = sorted((DATA / "shards").glob("*.bin"))
total_tokens = shard_token_count()
print(f"Шардов: {len(shards)}")
print(f"Всего токенов: {total_tokens:,} (~{total_tokens / 1e9:.2f} млрд)")
print("Токенизатор:", TOKENIZER)

In [ ]:
# Запуск обучения Seto-Small (~500M)
# train.py автоматически подхватывает последний чекпойнт из OUTPUT, если сессия была перезапущена!
train_cmd = [
    "torchrun", "--standalone", f"--nproc_per_node={GPU_COUNT}",
    "scripts/train.py",
    "--stage", "pretrain",
    "--model-config", "small",
    "--data-dir", DATA / "shards",
    "--output-dir", OUTPUT,
    "--tokenizer", TOKENIZER,
    "--batch-size", "2",
    "--grad-accum", "8",
    "--seq-len", "1024",
    "--lr", "3e-4",
    "--warmup-steps", "1000" if RUN_MODE == "full" else "100",
    "--max-steps", str(MAX_STEPS),
    "--save-every", str(SAVE_EVERY),
    "--log-every", "10",
    "--fp16",
]

print("Команда запуска:")
print(" ".join(map(str, train_cmd)))
run(*train_cmd, cwd=REPO)

In [ ]:
# Проверка чекпойнтов и свободного места
final_zip = OUTPUT / "final_pretrain.zip"
checkpoints = sorted((OUTPUT / "checkpoints_pretrain").glob("seto_step_*.zip"))
print("Последний чекпойнт:", checkpoints[-1] if checkpoints else "нет")
print("Финальный экспорт:", final_zip if final_zip.exists() else "еще не сформирован")

usage = shutil.disk_usage(WORK)
print(f"Свободно на диске: {usage.free / 1024**3:.2f} GiB")

## Что делать после завершения:
1. Скачать или подключить `final_pretrain.zip` к следующему блокноту `kaggle-seto-sft-1.ipynb`.
2. Если сессия Kaggle остановилась по тайм-ауту (до 12 часов), просто запусти ячейки снова: чекпойнты автоматически подхватятся!